# Rohit TextCNN walkthrough
Run this notebook after placing the full project folder in the environment. These cells are intentionally unexecuted; measured outputs belong to saved runs. Use a GPU runtime for full training if available.

In [ ]:
from pathlib import Path
# If opened inside notebooks/, move to the project root.
import os
if not Path("pyproject.toml").exists() and Path("../pyproject.toml").exists():
    os.chdir("..")
assert Path("pyproject.toml").exists(), "Change to the Rohit_DL_Project folder first"


In [ ]:
%pip install -e ".[dev]"


In [ ]:
!python -m pytest -q


## Check the pipeline
This synthetic integration run does not measure IMDB performance. Use a new output folder if one already exists.

In [ ]:
!rohit-dl smoke --output runs/notebook_smoke


## Prepare common datasets
Run once. Keep the exported splits and share their manifest with the team. Skip this cell if data already exists.

In [ ]:
!rohit-dl prepare --output data --ood yelp


## Train the full prescribed model
The best validation loss selects the checkpoint. Test labels never enter training.

In [ ]:
!rohit-dl train --data data --run runs/textcnn_seed42 --config configs/baseline.json


In [ ]:
!rohit-dl evaluate --run runs/textcnn_seed42 --input data/test.jsonl --output runs/clean_seed42


## Corruption robustness
Review the exported semantic review CSV before claiming label-preserving attacks. These are random corruptions.

In [ ]:
import nltk
Path(".cache/nltk_data").mkdir(parents=True, exist_ok=True)
nltk.download("wordnet", download_dir=".cache/nltk_data")
nltk.download("averaged_perceptron_tagger_eng", download_dir=".cache/nltk_data")
os.environ["NLTK_DATA"] = str(Path(".cache/nltk_data").resolve())


In [ ]:
!rohit-dl perturb --input data/test.jsonl --output data/perturbed --rates 0.05 0.10 0.20
!rohit-dl robustness --run runs/textcnn_seed42 --clean data/test.jsonl --perturbed data/perturbed --ood data/ood.jsonl --output runs/robustness_seed42


In [ ]:
import json
print(json.dumps(json.loads(Path("runs/robustness_seed42/summary.json").read_text()), indent=2))
from IPython.display import Image, display
display(Image(filename="runs/robustness_seed42/robustness.png"))


In [ ]:
!rohit-dl predict --run runs/textcnn_seed42 --text "The acting was excellent, but the story was not convincing."


## Preserve the evidence
Save the run folder including best.pt, vocab.json, config.json, provenance and history. Copy only small verified summaries and plots to reports for GitHub. Never present smoke scores or the interim projections as measured IMDB scores.